In [ ]:
# Libraries spaCy and NLTK
!pip install -q spacy nltk
!python -m spacy download it_core_news_sm




In [ ]:
from google.colab import files #upload dataset
uploaded = files.upload()

In [ ]:
from google.colab import files #upload module to clean names
uploaded = files.upload()

In [ ]:
import removename as rn #import module

In [ ]:
import sys
sys.path.append('/content')


import pandas as pd


# dataset
df = pd.read_csv("1.csv")

# create new column without names
df["no_names"] = df["comment_message"].apply(lambda x: rn.remove_user_name(str(x)))

# show result
print(df[["comment_message", "no_names"]].head())

# save and download
df.to_csv("1_no_names.csv", index=False)
files.download("1_no_names.csv")

Stop words removal:

In [ ]:
from google.colab import files #upload module to remove stopwords
uploaded = files.upload()

In [ ]:
# import libraries and module
import sys
sys.path.append('/content')

import cleantext3 as cl
import pandas as pd
from google.colab import files

# read the file
df = pd.read_csv("1_no_names.csv")

# new clean column
df["comment_message_clean"] = df["no_names"].apply(lambda x: cl.clean_italian_text(str(x)))

# show result
print(df[["no_names", "comment_message_clean"]].head())

# save and dowloand
df.to_csv("1_clean.csv", index=False)
files.download("1_clean.csv")


MOST FREQUENT WORDS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# load clean CSV
df = pd.read_csv("1_clean.csv")

# extract all words
all_text = ' '.join(df["comment_message_clean"].astype(str))

# clean "nan" or empty strings
all_text = all_text.replace("nan", "").strip()

# split into words
all_words = all_text.split()

# count frequencies
word_freq = Counter(all_words)

# select top N words
top_n = 20
most_common = word_freq.most_common(top_n)

# split words and counts
if most_common:
    words, counts = zip(*most_common)
else:
    words, counts = [], []

# create bar chart
plt.figure(figsize=(12, 6))
plt.bar(words, counts, color='skyblue')
plt.title(f"Most {top_n} frequent words", fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Frequency")
plt.ylim(0, 700)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# add number labels on top of bars
for i, v in enumerate(counts):
    plt.text(i, v + 0.3, str(v), ha='center', fontsize=9)

plt.tight_layout()

# SAVE TO PDF
plt.savefig("most_frequent_words_1.pdf", format="pdf", bbox_inches='tight')
print("✅ Chart saved as: most_frequent_words_1.pdf")

plt.show()

BI GRAMS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import spacy
from nltk.util import ngrams
import nltk
import re

# download the tokenizer if not already present
nltk.download('punkt', quiet=True)

# load Italian spaCy model
nlp = spacy.load("it_core_news_sm")

# load the cleaned CSV
df = pd.read_csv("1_clean.csv")

# join all cleaned text into a single string
all_text = ' '.join(df["comment_message_clean"].astype(str))
all_text = all_text.replace("nan", "").strip()

# tokenize the text (without lemmatization)
doc = nlp(all_text)

# extract words (excluding stopwords, punctuation, and empty spaces)
tokens = [
    token.text.lower()
    for token in doc
    if not token.is_stop and not token.is_punct and token.text.strip() != ""
]

# create bi-grams (pairs of consecutive words)
bigrams_list = list(ngrams(tokens, 2))

# count bi-gram frequencies
bigram_freq = Counter(bigrams_list)

# select the top N most frequent bi-grams
top_n = 20
most_common_bigrams = bigram_freq.most_common(top_n)

# prepare data for the plot
if most_common_bigrams:
    bigram_words = [' '.join(bigram) for bigram, _ in most_common_bigrams]
    counts = [count for _, count in most_common_bigrams]
else:
    bigram_words, counts = [], []

# create horizontal bar chart
plt.figure(figsize=(12, 6))

# use [::-1] to display the highest frequency at the top
plt.barh(bigram_words[::-1], counts[::-1], color='mediumslateblue')

plt.title("Top 20 bi-grams", fontsize=16)
plt.xlabel("Frequency")
plt.ylabel("Bi-gram")
plt.xlim(0, 60)
plt.grid(axis='x', linestyle='--', alpha=0.5)

# save the chart to a PDF file
# bbox_inches='tight' prevents labels from being cropped
plt.savefig("top_20_bigrams_1.pdf", format="pdf", bbox_inches='tight')
print("Chart saved as: top_20_bigrams_1.pdf")

plt.show()

lunghezza frasi

In [ ]:
df = df.dropna(subset=['comment_message'])

# count the number of words in each comment
df['sentence_length'] = df['comment_message'].apply(lambda x: len(x.split()))

# compute descriptive statistics
stats = {
    'num_comments': len(df),
    'mean_words': df['sentence_length'].mean(),
    'median_words': df['sentence_length'].median(),
    'std_dev': df['sentence_length'].std(),
    'min_words': df['sentence_length'].min(),
    'max_words': df['sentence_length'].max(),
    'quantile_25': df['sentence_length'].quantile(0.25),
    'quantile_75': df['sentence_length'].quantile(0.75),
    'variance': df['sentence_length'].var(),
}

# display results
print("Sentence Length Statistics (word count):\n")
for k, v in stats.items():
    print(f"{k:20}: {v}")